# 01 · LoCoMo 데이터셋

LoCoMo = **두 사람이 몇 달 동안 나눈 긴 대화** + **그 내용을 묻는 시험 문제(QA)**.

memory 시스템 평가는 3단계다:

1. 대화를 시스템에 넣는다
2. 시험 문제를 풀게 한다
3. 답을 채점한다

이 노트북에서는 재료인 **데이터를 직접 열어본다**.
(데이터가 없다면 먼저: `uv run scripts/fetch_data.py`)

In [1]:
from memlab.data import Category, load_locomo

samples = load_locomo()

print(f"대화:      {len(samples)}편")
print(f"발화(턴):  {sum(s.num_turns for s in samples):,}개")
print(f"시험 문제: {sum(len(s.qa) for s in samples):,}개")

대화:      10편
발화(턴):  5,882개
시험 문제: 1,986개


## 대화는 이렇게 생겼다

In [2]:
s = samples[0]
print(f"{s.sample_id}: {s.speaker_a} & {s.speaker_b}의 대화")
print(f"세션 {len(s.sessions)}개 / 발화 {s.num_turns}개")
print(f"{s.sessions[0].date_time}  부터")
print(f"{s.sessions[-1].date_time}  까지\n")

for t in s.sessions[0].turns[:4]:
    print(f"[{t.dia_id}] {t.speaker}: {t.text[:60]}")

conv-26: Caroline & Melanie의 대화
세션 19개 / 발화 419개
1:56 pm on 8 May, 2023  부터
9:55 am on 22 October, 2023  까지

[D1:1] Caroline: Hey Mel! Good to see you! How have you been?
[D1:2] Melanie: Hey Caroline! Good to see you! I'm swamped with the kids & w
[D1:3] Caroline: I went to a LGBTQ support group yesterday and it was so powe
[D1:4] Melanie: Wow, that's cool, Caroline! What happened that was so awesom


- **세션** = 날짜가 찍힌 하루치 대화. 한 대화는 세션 19~35개로 이뤄진다.
- **dia_id** = 발화의 주소. `D3:7` = 3번째 세션의 7번째 발화.
- 사진을 보내는 발화도 있다(전체의 21%) → 시스템에게는 **캡션 텍스트**로 보인다:

In [3]:
t = next(t for sess in s.sessions for t in sess.turns if t.has_image)
print(f"[{t.dia_id}] {t.speaker}: {t.text}")
print(f"   (사진) → 캡션: {t.blip_caption!r}")

[D1:5] Caroline: The transgender stories were so inspiring! I was so happy and thankful for all the support.
   (사진) → 캡션: 'a photo of a dog walking past a wall with a painting of a woman'


## 시험 문제(QA)란

국어 독해 시험과 같은 구조다:

- **지문** = 대화 한 편 (몇 달치, 수백 개 발화)
- **문제** = 그 내용을 묻는 질문. 대화마다 105~260개, 전부 합쳐 1,986개
- **학생** = memory 시스템 — 대화를 발화 하나씩 다 읽은 뒤, 자기 기억만으로 답한다

대화 전체는 너무 길어서 통째로 기억할 수 없다. 그래서 **뭘 남기고 뭘 버릴지**가
memory 시스템의 실력이고, 이 1,986문제가 그 실력을 재는 자다.

## 문제는 5종류다

| cat | 이름 | 무엇을 시험하나 | 문항 수 |
|---|---|---|---|
| 1 | multi-hop | 여러 발화를 종합해야 답이 나온다 | 282 |
| 2 | temporal | "언제?" — 시간 기억 | 321 |
| 3 | open-domain | 대화 + 상식으로 추론 | 96 |
| 4 | single-hop | 발화 하나에서 바로 답 | 841 |
| 5 | adversarial | **일어난 적 없는 일** — 안 속아야 정답 | 446 |

종류별로 하나씩 보자 (근거 = 답이 나온 발화의 주소):

In [4]:
seen = set()
for sm in samples:
    for q in sm.qa:
        if q.category not in seen:
            seen.add(q.category)
            print(f"[cat{int(q.category)}] Q: {q.question}")
            print(f"       A: {q.answer}   (근거: {', '.join(q.evidence)})\n")
    if len(seen) == 5:
        break

[cat2] Q: When did Caroline go to the LGBTQ support group?
       A: 7 May 2023   (근거: D1:3)

[cat3] Q: What fields would Caroline be likely to pursue in her educaton?
       A: Psychology, counseling certification   (근거: D1:9, D1:11)

[cat1] Q: What did Caroline research?
       A: Adoption agencies   (근거: D2:8)

[cat4] Q: What did the charity race raise awareness for?
       A: mental health   (근거: D2:2)

[cat5] Q: What did Caroline realize after her charity race?
       A: None   (근거: D2:3)



cat5만 정답이 `None`이다. **"모른다"고 해야 정답**이기 때문.
그래서 cat5는 일반적인 채점(F1/BLEU)이 안 된다 — 챕터 02에서 따로 다룬다.

### 위 표의 이름들, 믿어도 되나?

데이터에는 카테고리가 **숫자로만** 들어 있다 (1~5). 이름이 맞는지 확인해 보자.

- multi-hop이 맞다면 → 근거(evidence)가 여러 개일 것이다
- temporal이 맞다면 → 답이 날짜일 것이다

In [5]:
import re

date_pat = re.compile(
    r"\b(19|20)\d{2}\b|january|february|march|april|may|june"
    r"|july|august|september|october|november|december",
    re.I,
)

print("cat | 근거 평균 | 답이 날짜")
for cat in Category:
    qs = [q for sm in samples for q in sm.qa if q.category == cat]
    ev = sum(len(q.evidence) for q in qs) / len(qs)
    dates = sum(1 for q in qs if q.answer and date_pat.search(q.answer)) / len(qs)
    print(f"  {int(cat)} |   {ev:.1f}개   |   {dates:4.0%}   {cat.name}")

cat | 근거 평균 | 답이 날짜
  1 |   3.1개   |     2%   MULTI_HOP
  2 |   1.2개   |    80%   TEMPORAL
  3 |   2.1개   |     0%   OPEN_DOMAIN
  4 |   1.1개   |     1%   SINGLE_HOP
  5 |   1.0개   |     0%   ADVERSARIAL


예상대로다. cat1만 근거가 평균 3개(→ multi-hop), cat2만 답의 80%가 날짜(→ temporal).

## 정리

- 대화 10편(발화 5,882개) + 시험 문제 1,986개
- 문제는 5종류: single-hop / multi-hop / temporal / open-domain / adversarial
- cat5(adversarial)만 정답이 없다 — "모른다"가 정답

**다음 — 02 · 채점**: "답이 맞았다"를 어떻게 숫자로 정하나 (F1, BLEU-1)